# 02 - Performance Review

## Purpose

Identify performance and maintainability risks in the completed capstone.

This notebook reviews repeated actions, wide joins, unnecessary columns, repeated reads, display overload, and query plans.

## Required evidence

- at least five performance or maintainability issues
- at least two `explain(True)` inspections
- one Silver transformation inspection
- one Gold aggregation or join inspection

## Main idea

Day 5 optimization should be targeted. The goal is not to rewrite the whole project, but to identify real risks and improve selected parts.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"

silver_grid_table = f"{catalog}.refined.silver_grid_events"
gold_market_table = f"{catalog}.analytics.gold_daily_market_summary"
gold_grid_table = f"{catalog}.analytics.gold_grid_incident_summary"
dashboard_table = f"{catalog}.analytics.gold_regional_operations_dashboard"

print("Silver grid table:", silver_grid_table)
print("Gold market table:", gold_market_table)
print("Gold grid table:", gold_grid_table)
print("Dashboard table:", dashboard_table)

In [0]:
performance_issues = [
    Row(
        issue_id="PERF-001",
        issue_type="Repeated actions",
        issue_found="Several notebooks call .count() multiple times on the same DataFrame.",
        risk="Each .count() can trigger a Spark job, increasing unnecessary execution work.",
        improvement="Store counts in variables and reuse them in logs and validation.",
        status="identified",
    ),
    Row(
        issue_id="PERF-002",
        issue_type="Wide joins",
        issue_found="Some joins originally carried duplicate or unnecessary technical columns.",
        risk="Wide DataFrames make joins harder to debug and can increase processing overhead.",
        improvement="Select business-relevant columns before joins.",
        status="improved",
    ),
    Row(
        issue_id="PERF-003",
        issue_type="Ambiguous columns",
        issue_found="Joined DataFrames can contain repeated column names such as incident_count or region.",
        risk="Ambiguous references can cause runtime errors or unclear output schemas.",
        improvement="Rename or select columns explicitly before joining.",
        status="improved",
    ),
    Row(
        issue_id="PERF-004",
        issue_type="Display overload",
        issue_found="Development notebooks use display() several times for inspection.",
        risk="Useful for learning, but too many displays can make notebooks noisy and slower to review.",
        improvement="Keep only representative displays and summarize row counts in logs.",
        status="documented",
    ),
    Row(
        issue_id="PERF-005",
        issue_type="Late validation",
        issue_found="If validation only happens at the end, issues may be harder to trace.",
        risk="Broken assumptions can be hidden inside later aggregations.",
        improvement="Validate key row counts, nulls, and duplicate grains immediately after writes.",
        status="improved",
    ),
]

performance_issues_df = spark.createDataFrame(performance_issues)

display(performance_issues_df)

In [0]:
silver_grid_df = spark.table(silver_grid_table)

silver_grid_review_df = (
    silver_grid_df
    .select(
        "event_id",
        "event_day",
        "region",
        "asset_id",
        "severity_band",
        "duration_minutes",
    )
    .filter(F.col("event_day").isNotNull())
    .filter(F.col("duration_minutes") >= 0)
)

print("Explain plan: Silver grid review DataFrame")
silver_grid_review_df.explain(True)

In [0]:
gold_grid_df = spark.table(gold_grid_table)

gold_grid_review_df = (
    gold_grid_df
    .groupBy("event_day", "region")
    .agg(
        F.sum("incident_count").alias("total_incident_count"),
        F.sum("elevated_incident_count").alias("total_elevated_incident_count"),
        F.sum("total_duration_minutes").alias("total_duration_minutes"),
    )
)

print("Explain plan: Gold grid day-region aggregation")
gold_grid_review_df.explain(True)

In [0]:
market_df = spark.table(gold_market_table)
grid_day_region_df = gold_grid_review_df

dashboard_join_review_df = (
    market_df
    .select(
        "report_day",
        "region",
        "avg_price_eur_mwh",
        "total_volume_mwh",
        "is_high_market_price",
    )
    .join(
        grid_day_region_df,
        (market_df["report_day"] == grid_day_region_df["event_day"])
        & (market_df["region"] == grid_day_region_df["region"]),
        how="left"
    )
)

print("Explain plan: Gold dashboard-style join")
dashboard_join_review_df.explain(True)

In [0]:
optimization_observations = [
    Row(
        observation="Silver review uses column pruning",
        explanation="Only columns needed for event review are selected before filtering.",
        benefit="Reduces unnecessary columns carried into later operations.",
    ),
    Row(
        observation="Gold aggregation uses a clear reporting grain",
        explanation="The aggregation groups by event_day and region.",
        benefit="Makes the output easier to validate and safer for dashboard use.",
    ),
    Row(
        observation="Dashboard-style join should select needed columns first",
        explanation="Market and grid data are reduced before joining.",
        benefit="Avoids carrying unnecessary columns into the join.",
    ),
]

optimization_observations_df = spark.createDataFrame(optimization_observations)

display(optimization_observations_df)

In [0]:
print("Performance review completed.")
print("At least five risks or issues were identified.")
print("At least two explain(True) inspections were run.")
print("Targeted improvements are documented for Day 5 hardening.")